# 05_GSE254249_qc_selection

**Thesis Methods section(s): 4.1.1, 4.1.2, 4.1.3**

**Reads:** GSE254249 matrix, barcodes, features and metadata (GEO).

**Writes:** GSE254249_BL_CD8_STRICT_postQC_scvi_rawcounts.h5ad

**Notes:** CD8 cells come from the author-provided subCluster labels (entries containing '-CD8-'), plus the double-positive subcluster TE05-T-DP. Restricted to pre-treatment (baseline) samples and to the PBMC and Rectum_T compartments.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


In [ ]:
import os, gzip
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

DATA_DIR = f"{DATA_ROOT}/scVI/9 GSE254249"  # change if needed

barcodes_fp = os.path.join(DATA_DIR, "GSE254249_scRNA_barcodes.tsv.gz")
features_fp = os.path.join(DATA_DIR, "GSE254249_scRNA_features.tsv.gz")
matrix_fp   = os.path.join(DATA_DIR, "GSE254249_scRNA_matrix.mtx.gz")
meta_fp     = os.path.join(DATA_DIR, "GSE254249_scRNA_metadata.tsv.gz")

barcodes = pd.read_csv(barcodes_fp, header=None, sep="\t")[0].astype(str).values
features = pd.read_csv(features_fp, header=None, sep="\t")
features.columns = [f"col{i}" for i in range(features.shape[1])]
gene_symbols = (features["col1"] if features.shape[1] >= 2 else features["col0"]).astype(str).values

meta = pd.read_csv(meta_fp, sep="\t")
print("barcodes:", len(barcodes), "genes:", len(gene_symbols))
print("meta shape:", meta.shape)
print("meta columns sample:", list(meta.columns[:30]))

In [ ]:
# Make sure barcodes are indexed by the correct column
barcode_col = "Unnamed: 0"
meta[barcode_col] = meta[barcode_col].astype(str)

print("Tissue counts:")
print(meta["Tissue"].astype(str).value_counts().head(20), "\n")

print("SampleTimePoint counts:")
print(meta["SampleTimePoint"].astype(str).value_counts().head(20), "\n")

print("Ident (top 30):")
print(meta["Ident"].astype(str).value_counts().head(30), "\n")

print("subCluster (top 30):")
print(meta["subCluster"].astype(str).value_counts().head(30))

In [ ]:
import numpy as np
import pandas as pd

barcode_col = "Unnamed: 0"

meta2 = meta.copy()
meta2[barcode_col] = meta2[barcode_col].astype(str)

# Pre-treatment baseline
is_pre = meta2["SampleTimePoint"].astype(str).eq("BL")

# Keep both compartments
is_keep_tissue = meta2["Tissue"].astype(str).isin(["PBMC", "Rectum_T"])

# CD8 definition from subCluster
sc_ = meta2["subCluster"].astype(str)

# Main CD8 groups are TE..-CD8-...
is_cd8 = sc_.str.contains(r"-CD8-", regex=True)

# Optional: include TE05-T-DP (comment out if you want strict CD8 only)
include_T_DP = True
if include_T_DP:
    is_cd8 = is_cd8 | sc_.str.contains(r"TE05-T-DP", regex=True)

sel = is_pre & is_keep_tissue & is_cd8

print("Selected cells:", int(sel.sum()))
print(meta2.loc[sel, "Tissue"].value_counts())
print(meta2.loc[sel, "subCluster"].value_counts().head(25))
print("Patients:", meta2.loc[sel, "PatientID"].nunique())

In [ ]:
# Build barcode → index map (global indices from barcodes.tsv)
barcode_to_idx = {b: i for i, b in enumerate(barcodes)}

# Extract selected barcodes
sel_barcodes = meta2.loc[sel, "Unnamed: 0"].values.astype(str)

# Map to indices in barcodes array
sel_idx = np.array(
    [barcode_to_idx[b] for b in sel_barcodes if b in barcode_to_idx],
    dtype=np.int64
)

print("Selected barcodes:", len(sel_barcodes))
print("Mapped to indices:", sel_idx.size)

if sel_idx.size == 0:
    raise ValueError("No selected barcodes matched barcodes.tsv — formatting mismatch.")

In [ ]:
import gzip
import numpy as np
import scipy.sparse as sp
import anndata as ad

# sel_idx_sorted = np.sort(sel_idx)  # you already computed this earlier
sel_idx_sorted = np.sort(sel_idx).astype(np.int64)

n_sel_cells = sel_idx_sorted.size
n_genes = len(gene_symbols)

rows = []
cols = []
data = []

def _skip_mm_comments(fh):
    line = fh.readline()
    while line.startswith(b"%"):
        line = fh.readline()
    return line

# Pointer-based membership:
# We iterate through MTX entries and advance a pointer through sel_idx_sorted
# BUT j in MTX is not sorted, so pointer method won't work unless MTX is sorted by column.
# So we do a fast membership with a boolean mask instead (O(1) lookup via numpy array).

# Boolean mask over all cells (619635) -> True for selected
mask = np.zeros(len(barcodes), dtype=np.bool_)
mask[sel_idx_sorted] = True

# Map original cell index -> new index via an int32 array (fast)
# For non-selected cells, value stays -1
orig_to_new = np.full(len(barcodes), -1, dtype=np.int32)
orig_to_new[sel_idx_sorted] = np.arange(n_sel_cells, dtype=np.int32)

with gzip.open(matrix_fp, "rb") as fh:
    header = fh.readline()
    if not header.startswith(b"%%MatrixMarket"):
        raise ValueError("Not a MatrixMarket file.")

    shape_line = _skip_mm_comments(fh)
    nrow, ncol, nnz = map(int, shape_line.decode().strip().split())
    print("MTX shape (genes x cells):", (nrow, ncol), "nnz:", nnz)

    kept = 0
    for line in fh:
        i_str, j_str, v_str = line.split()
        gene_i = int(i_str) - 1
        cell_j = int(j_str) - 1

        if mask[cell_j]:
            new_j = orig_to_new[cell_j]
            rows.append(new_j)                  # new cell row
            cols.append(gene_i)                 # gene column
            data.append(np.int32(float(v_str))) # counts
            kept += 1

print("✅ Kept nnz:", kept)

X = sp.coo_matrix(
    (np.array(data, dtype=np.int32),
     (np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32))),
    shape=(n_sel_cells, n_genes)
).tocsr()

adata_cd8 = ad.AnnData(X=X)
adata_cd8.obs_names = barcodes[sel_idx_sorted]
adata_cd8.var_names = gene_symbols
adata_cd8.var_names_make_unique()

# Attach metadata (barcode col is Unnamed: 0)
meta_sub = meta2.set_index("Unnamed: 0").loc[adata_cd8.obs_names].copy()
adata_cd8.obs = adata_cd8.obs.join(meta_sub)

adata_cd8.layers["counts"] = adata_cd8.X.copy()

print(adata_cd8)
print("Cells by Tissue:\n", adata_cd8.obs["Tissue"].value_counts())
print("Cells by subCluster (top 10):\n", adata_cd8.obs["subCluster"].value_counts().head(10))

In [ ]:
import time
print("alive", time.time())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = adata_cd8.obs[["PatientID","Tissue"]].copy()

counts = (
    df.groupby(["PatientID","Tissue"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

ax = counts.plot(kind="bar", figsize=(12,6))
ax.set_title("GSE254249 (BL) CD8 cells per patient and tissue")
ax.set_xlabel("PatientID")
ax.set_ylabel("Number of cells")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fractions = counts.div(counts.sum(axis=1), axis=0)

ax = fractions.plot(kind="bar", figsize=(12,6))
ax.set_title("Fraction of CD8 cells per patient by tissue (PBMC vs Rectum_T)")
ax.set_xlabel("PatientID")
ax.set_ylabel("Fraction")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df = adata_cd8.obs[["Tissue","subCluster"]].copy()

comp = (
    df.groupby(["Tissue","subCluster"])
      .size()
      .unstack(fill_value=0)
)

# sort columns by total abundance
comp = comp.loc[:, comp.sum(axis=0).sort_values(ascending=False).index]

ax = comp.T.plot(kind="bar", figsize=(14,6))
ax.set_title("CD8 subCluster counts by tissue")
ax.set_xlabel("subCluster")
ax.set_ylabel("Cells")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
top = comp.sum(axis=0).sort_values(ascending=False).head(10).index
comp_top = comp[top]

ax = comp_top.T.plot(kind="bar", figsize=(12,5))
ax.set_title("Top 10 CD8 subClusters by tissue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df = adata_cd8.obs[["PatientID","subCluster"]].copy()

tab = (
    df.groupby(["PatientID","subCluster"])
      .size()
      .unstack(fill_value=0)
)

# Keep top 8 clusters, pool rest as "Other"
top = tab.sum(axis=0).sort_values(ascending=False).head(8).index
tab_small = tab[top].copy()
tab_small["Other"] = tab.drop(columns=top).sum(axis=1)

# Plot fractions
tab_frac = tab_small.div(tab_small.sum(axis=1), axis=0)

ax = tab_frac.plot(kind="bar", stacked=True, figsize=(14,6))
ax.set_title("Per-patient CD8 subCluster composition (fractions)")
ax.set_xlabel("PatientID")
ax.set_ylabel("Fraction")
plt.xticks(rotation=45, ha="right")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

df = adata_cd8.obs[["Tissue","subCluster"]].copy()

ct = (
    df.groupby(["Tissue","subCluster"])
      .size()
      .unstack(fill_value=0)
)

# Convert to fractions within each tissue
frac = ct.div(ct.sum(axis=1), axis=0)

eps = 1e-9
enrich = np.log2((frac.loc["Rectum_T"] + eps) / (frac.loc["PBMC"] + eps)).sort_values()

plt.figure(figsize=(8,6))
plt.barh(enrich.index, enrich.values)
plt.axvline(0, linestyle="--")
plt.title("Tumor vs PBMC enrichment (log2 ratio) per CD8 subCluster")
plt.xlabel("log2( frac_tumor / frac_PBMC )")
plt.tight_layout()
plt.show()

In [ ]:
adata_cd8.uns["notes"] = {
    "dataset": "GSE254249",
    "timepoint": "BL only",
    "cells_selected": "CD8-related subClusters (including TE05-T-DP)",
    "tissues": ["PBMC", "Rectum_T"],
    "n_cells": int(adata_cd8.n_obs),
    "n_genes": int(adata_cd8.n_vars)
}

In [ ]:
import os

raw_fp = os.path.join(DATA_DIR, "GSE254249_BL_CD8_preQC_rawcounts.h5ad")
adata_cd8.write(raw_fp)

print("✅ Pre-QC raw object saved at:")
print(raw_fp)

In [ ]:
import scanpy as sc
import os

DATA_DIR = f"{DATA_ROOT}/scVI/9 GSE254249"

raw_fp = os.path.join(DATA_DIR, "GSE254249_BL_CD8_preQC_rawcounts.h5ad")

adata_cd8 = sc.read_h5ad(raw_fp)

print(adata_cd8)

In [ ]:
import scanpy as sc
import numpy as np

adata = adata_cd8.copy()

# Ensure raw counts are stored in a layer
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

# QC metrics computed from raw counts
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], layer="counts", inplace=True)

# Quick look
adata.obs[["total_counts","n_genes_by_counts","pct_counts_mt"]].describe()

In [ ]:
sc.pl.violin(
    adata,
    ["total_counts","n_genes_by_counts","pct_counts_mt"],
    groupby="Tissue",
    jitter=0.4,
    multi_panel=True
)

sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="Tissue")
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", color="Tissue")

In [ ]:
adata = adata[
    (adata.obs["n_genes_by_counts"] >= 300) &
    (adata.obs["total_counts"] >= 500) &
    (adata.obs["pct_counts_mt"] <= 20)
].copy()

print("After filter:", adata.shape)

In [ ]:
# Verify counts layer exists
print("Counts layer present:", "counts" in adata.layers)

# Ensure scVI will use raw counts
adata.X = adata.layers["counts"].copy()

In [ ]:
# Optional cleanup
for col in ["mt"]:
    if col in adata.var.columns:
        adata.var.drop(columns=[col], inplace=True)

In [ ]:
import os

scvi_fp = os.path.join(DATA_DIR, "GSE254249_BL_CD8_postQC_scvi_rawcounts.h5ad")
adata.write(scvi_fp)

print("✅ scVI-ready object saved at:")
print(scvi_fp)
print(adata)

Import scanpy and reconstruct the path

In [ ]:
import scanpy as sc
import os

DATA_DIR = f"{DATA_ROOT}/scVI/9 GSE254249"

scvi_fp = os.path.join(DATA_DIR, "GSE254249_BL_CD8_postQC_scvi_rawcounts.h5ad")

In [ ]:
adata = sc.read_h5ad(scvi_fp)

print("Loaded object:")
print(adata)

In [ ]:
adata.obs["Tissue"].value_counts()